In [1]:
from collections import defaultdict

corpus = [
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit near me",
    "best places to visit during summer",
    "best hotels in chennai",
    "best restaurants in india"
]

D = 0.75

# Store 5-gram counts
fivegram = defaultdict(int)

# Store 4-word history counts
history = defaultdict(int)

# ------------------------
# Build 5-gram Model
# ------------------------

for sentence in corpus:

    words = sentence.lower().split()

    for i in range(len(words)-4):

        hist = tuple(words[i:i+4])
        nxt = words[i+4]

        fivegram[(hist, nxt)] += 1
        history[hist] += 1

# ------------------------
# Continuation Counts
# ------------------------

continuation = defaultdict(set)

for (hist, nxt) in fivegram:
    continuation[nxt].add(hist)

# ------------------------
# Followers
# ------------------------

followers = defaultdict(set)

for (hist, nxt) in fivegram:
    followers[hist].add(nxt)

total_unique_fivegrams = len(fivegram)

# ------------------------
# Kneser-Ney Probability
# ------------------------

def kneser_ney(history_words, current_word):

    history_words = tuple(history_words)

    count = fivegram.get((history_words, current_word), 0)

    history_count = history.get(history_words, 0)

    if history_count == 0:
        return len(continuation[current_word]) / max(total_unique_fivegrams,1)

    first = max(count - D, 0) / history_count

    lamb = (D * len(followers[history_words])) / history_count

    cont_prob = len(continuation[current_word]) / total_unique_fivegrams

    return first + lamb * cont_prob

# ------------------------
# User Query
# ------------------------

query = input("Enter Search Query: ").lower()

words = query.split()

history_words = words[-4:]

print("\nAutocomplete Suggestions:")

for sentence in corpus:

    if sentence.startswith(query):

        print("-", sentence[len(query):].strip())

print("\nNext Word Prediction:")

vocabulary = set()

for sentence in corpus:
    vocabulary.update(sentence.split())

scores = {}

for word in vocabulary:

    scores[word] = kneser_ney(history_words, word)

prediction = sorted(scores.items(), key=lambda x:x[1], reverse=True)

for word, score in prediction[:5]:
    print(word, ":", round(score,3))


Autocomplete Suggestions:
- best places to visit in india
- best places to visit in chennai
- best places to visit near me
- best places to visit during summer
- best hotels in chennai
- best restaurants in india

Next Word Prediction:
me : 0.143
during : 0.143
in : 0.143
india : 0.143
summer : 0.143
